# 🎾 Padel Analytics — Clustering & Time Series Forecasting
## Projet d'Analyse de Performances — Sections E & F

---
**Dataset :** `fact_performanceF.csv` — 480 lignes, 26 colonnes  
**Joueurs :** 4 joueurs (ID: 1, 3, 5, 12) | **Années :** 2022–2025  
**Sections couvertes :**
- Section A : Data Preparation & Feature Engineering
- Section B : Model Understanding (explications théoriques intégrées)
- Section E : Clustering (K-Means, DBSCAN, Clustering Hiérarchique)
- Section F : Time Series Forecasting (ARIMA, Prophet, XGBoost TS)
---

## 📦 0. Installation des dépendances

In [ ]:
# Installer les librairies nécessaires (à exécuter une seule fois)
!pip install prophet scikit-learn xgboost statsmodels matplotlib seaborn pandas numpy scipy -q

## 📚 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Clustering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.impute import SimpleImputer

# Time Series
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import itertools

# Prophet
from prophet import Prophet

# XGBoost
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Style
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
sns.set_palette(PALETTE)

print('✅ Tous les imports réussis !')

---
# 🧹 SECTION A — Data Preparation & Feature Engineering

## A.1 — Chargement & exploration initiale

In [ ]:
df = pd.read_csv('fact_performanceF.csv')

print(f'📐 Shape : {df.shape}')
print(f'📅 Années : {sorted(df["Year"].unique())}')
print(f'👤 Joueurs (ID) : {sorted(df["ID_player"].unique())}')
print(f'🏷️ Tiers : {df["Tier"].unique().tolist()}')
print()
df.head()

In [ ]:
# Informations sur les types et valeurs manquantes
info_df = pd.DataFrame({
    'Type': df.dtypes,
    'Nulls': df.isnull().sum(),
    'Null%': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique': df.nunique()
})
print(info_df[info_df['Nulls'] > 0])

In [ ]:
# Visualisation des valeurs manquantes
fig, ax = plt.subplots(figsize=(12, 4))
missing = df.isnull().sum()
missing = missing[missing > 0]
bars = ax.bar(missing.index, missing.values, color='#FF5722', alpha=0.8, edgecolor='black')
for bar, val in zip(bars, missing.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('Valeurs manquantes par colonne', fontsize=14, fontweight='bold')
ax.set_ylabel('Nombre de valeurs nulles')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## A.2 — Nettoyage & Imputation

In [ ]:
df_clean = df.copy()

# Colonnes avec valeurs manquantes
cols_with_null = ['Nombre_de_spectateurs', 'scheduled_matches_count',
                  'match_reservations_count', 'likes']

# Imputation par la médiane (robuste aux outliers)
for col in cols_with_null:
    median_val = df_clean[col].median()
    df_clean[col].fillna(median_val, inplace=True)
    print(f'  → {col} : imputé avec médiane = {median_val:.0f}')

# Encodage de Tier (Top10=1, Hors Top=0)
df_clean['Tier_encoded'] = (df_clean['Tier'] == 'Top10').astype(int)

print(f'\n✅ Dataset nettoyé — Shape : {df_clean.shape}')
print(f'Valeurs nulles restantes : {df_clean.isnull().sum().sum()}')

## A.3 — Feature Engineering

In [ ]:
# === Feature Engineering ===

# 1. Taux de victoire
df_clean['win_rate'] = df_clean['victoires'] / df_clean['matchs_joues'].replace(0, np.nan)
df_clean['win_rate'].fillna(0, inplace=True)

# 2. Points par match
df_clean['points_per_match'] = df_clean['points'] / df_clean['matchs_joues'].replace(0, np.nan)
df_clean['points_per_match'].fillna(0, inplace=True)

# 3. Taux de remplissage (réservations / matchs programmés)
df_clean['fill_rate'] = (df_clean['match_reservations_count'] /
                         df_clean['scheduled_matches_count'].replace(0, np.nan))
df_clean['fill_rate'].fillna(0, inplace=True)

# 4. ROI estimé (prize money vs coût)
df_clean['prize_money_avg'] = (df_clean['prize_money_min'] + df_clean['prize_money_max']) / 2
df_clean['cost_avg'] = (df_clean['real_cost_min'] + df_clean['real_cost_max']) / 2
df_clean['roi'] = df_clean['prize_money_avg'] / df_clean['cost_avg'].replace(0, np.nan)
df_clean['roi'].fillna(0, inplace=True)

# 5. Engagement social normalisé
df_clean['social_score'] = np.log1p(df_clean['likes']) + np.log1p(df_clean['Abonnes_Instagram_Novembre_2025'])

print('✅ Features créées :')
new_features = ['win_rate', 'points_per_match', 'fill_rate', 'roi', 'social_score']
print(df_clean[new_features].describe().round(3))

---
# 📖 SECTION B — Compréhension des modèles

## B.1 — Modèles de Clustering : Théorie & Intuition

### 🔵 K-Means
**Intuition :** K-Means partitionne les données en K clusters en minimisant la somme des distances au carré entre chaque point et le centroïde (centre) de son cluster.

**Algorithme :**
1. Initialiser K centroïdes aléatoirement (ou via K-Means++)
2. Assigner chaque point au centroïde le plus proche
3. Recalculer les centroïdes
4. Répéter jusqu'à convergence

**Paramètres clés :** `n_clusters` (K), `init`, `max_iter`, `random_state`  
**Hypothèses :** Clusters sphériques, taille similaire, données normalisées  
**Limitations :** Sensible aux outliers, K à choisir à l'avance, ne gère pas les formes non-convexes  
**Utilisation ici :** Segmenter les tournois/joueurs par profil de performance

---
### 🔴 DBSCAN (Density-Based Spatial Clustering)
**Intuition :** DBSCAN groupe des points qui sont proches (denses) ensemble et marque les points isolés comme **outliers/bruit**.

**Algorithme :**
1. Pour chaque point, compter les voisins dans un rayon `eps`
2. Un point avec ≥ `min_samples` voisins = **core point**
3. Les points atteignables depuis un core point = même cluster
4. Points non atteignables = bruit (-1)

**Paramètres clés :** `eps` (rayon), `min_samples` (densité minimum)  
**Hypothèses :** Clusters denses, pas besoin de connaître K à l'avance  
**Limitations :** Sensible à `eps`, mauvais sur données haute dimension  
**Utilisation ici :** Détecter des tournois atypiques / outliers

---
### 🟢 Clustering Hiérarchique (Agglomératif)
**Intuition :** Construit un arbre (dendrogramme) en fusionnant progressivement les points/clusters les plus proches.

**Algorithme :**
1. Commencer : chaque point = un cluster
2. Fusionner les 2 clusters les plus proches
3. Répéter jusqu'à avoir un seul cluster
4. Couper le dendrogramme au niveau souhaité → K clusters

**Paramètres clés :** `n_clusters`, `linkage` (ward/complete/average/single)  
**Avantage :** Pas besoin de K à l'avance, visualisation intuitive  
**Limitations :** Complexité O(n²), pas de réaffectation possible  
**Utilisation ici :** Explorer les similarités entre joueurs

## B.2 — Modèles de Time Series : Théorie & Intuition

### 📈 ARIMA (AutoRegressive Integrated Moving Average)
**Intuition :** ARIMA modélise une série temporelle en fonction de ses propres valeurs passées (AR), de ses erreurs passées (MA) et en la rendant stationnaire via différenciation (I).

**Paramètres :** `p` (AR order), `d` (différenciation), `q` (MA order)  
**Hypothèses :** Série stationnaire après différenciation, résidus en bruit blanc  
**Limitations :** Linéaire, ne capture pas les saisonnalités complexes  
**Utilisation ici :** Prédire les points / classement par joueur

---
### 🔮 Prophet (Facebook/Meta)
**Intuition :** Prophet décompose la série en tendance + saisonnalité + effets calendaires. Conçu pour des séries **business** avec tendances changeantes et effets saisonniers annuels/hebdomadaires.

**Composantes :** `trend` (logistique ou linéaire) + `seasonality` (Fourier) + `holidays`  
**Avantages :** Robuste aux outliers, gère les données manquantes, interprétable  
**Limitations :** Moins précis sur très courtes séries, suppose tendance additive/multiplicative  
**Utilisation ici :** Prévision des points sur les prochaines saisons

---
### ⚡ XGBoost pour Time Series
**Intuition :** XGBoost est un algorithme de gradient boosting appliqué à la prévision temporelle en transformant le problème en régression supervisée avec des **lag features** (valeurs passées comme variables explicatives).

**Paramètres :** `n_estimators`, `max_depth`, `learning_rate`, `lag_features`  
**Avantages :** Capture les relations non-linéaires, très performant  
**Limitations :** Nécessite feature engineering temporel manuel, peut overfitter  
**Utilisation ici :** Prédire les gains en prize money

---
# 🔵 SECTION E — Clustering

## Objectifs :
1. **Segmenter les tournois** par profil de performance (K-Means)
2. **Détecter les tournois atypiques** (DBSCAN)
3. **Analyser les similitudes entre joueurs** (Hiérarchique)
4. Comparer les méthodes via Silhouette Score et Davies-Bouldin Index

## E.1 — Préparation des features pour le Clustering

In [ ]:
# Features numériques pertinentes pour le clustering de tournois
clustering_features = [
    'points', 'win_rate', 'points_per_match', 'classement_mondial',
    'Nombre_de_spectateurs', 'prize_money_avg', 'fill_rate', 'roi',
    'social_score', 'price', 'Tier_encoded'
]

X_cluster = df_clean[clustering_features].copy()

# Imputation finale (sécurité)
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_cluster)

# Normalisation (INDISPENSABLE pour K-Means et DBSCAN)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print(f'✅ Shape pour clustering : {X_scaled.shape}')
print(f'Features utilisées : {clustering_features}')

# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))
corr_matrix = pd.DataFrame(X_imputed, columns=clustering_features).corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, annot_kws={'size': 8})
ax.set_title('Matrice de corrélation — Features de clustering', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## E.2 — Modèle 1 : K-Means + Méthode du Coude + Silhouette

In [ ]:
# === MÉTHODE DU COUDE (Elbow Method) ===
inertias = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    silhouette_scores.append(sil)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Nombre de clusters K', fontsize=12)
axes[0].set_ylabel('Inertie (WCSS)', fontsize=12)
axes[0].set_title('📊 Méthode du Coude (Elbow Method)', fontsize=13, fontweight='bold')
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K optimal = 4')
axes[0].legend()

# Silhouette
axes[1].plot(list(k_range), silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Nombre de clusters K', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('📏 Silhouette Score par K', fontsize=13, fontweight='bold')
best_k = k_range.start + np.argmax(silhouette_scores)
axes[1].axvline(x=best_k, color='green', linestyle='--', alpha=0.7,
                label=f'Meilleur K = {best_k} ({max(silhouette_scores):.3f})')
axes[1].legend()

plt.suptitle('K-Means : Choix du nombre optimal de clusters', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n🎯 K optimal selon Silhouette : K = {best_k} (score = {max(silhouette_scores):.4f})')

In [ ]:
# === K-Means avec K optimal ===
K_OPTIMAL = best_k

kmeans = KMeans(n_clusters=K_OPTIMAL, init='k-means++', n_init=20, random_state=42)
df_clean['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

sil_km = silhouette_score(X_scaled, df_clean['cluster_kmeans'])
db_km = davies_bouldin_score(X_scaled, df_clean['cluster_kmeans'])

print(f'✅ K-Means avec K={K_OPTIMAL}')
print(f'   Silhouette Score : {sil_km:.4f} (plus proche de 1 = mieux)')
print(f'   Davies-Bouldin   : {db_km:.4f} (plus proche de 0 = mieux)')
print()
print('Distribution des clusters :')
print(df_clean['cluster_kmeans'].value_counts().sort_index())

In [ ]:
# === PCA 2D pour visualisation K-Means ===
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained_var = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# PCA plot
scatter_colors = plt.cm.tab10(np.linspace(0, 0.5, K_OPTIMAL))
for i in range(K_OPTIMAL):
    mask = df_clean['cluster_kmeans'] == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   label=f'Cluster {i}', s=60, alpha=0.8)

# Centroïdes en PCA
centroids_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
               marker='*', s=300, c='black', zorder=5, label='Centroïdes')
axes[0].set_xlabel(f'PC1 ({explained_var[0]*100:.1f}% variance)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({explained_var[1]*100:.1f}% variance)', fontsize=11)
axes[0].set_title(f'K-Means — Visualisation PCA 2D (K={K_OPTIMAL})', fontsize=13, fontweight='bold')
axes[0].legend()

# Profil des clusters
key_features = ['win_rate', 'points_per_match', 'prize_money_avg', 'roi', 'social_score']
cluster_profiles = df_clean.groupby('cluster_kmeans')[key_features].mean()

# Normaliser pour radar/heatmap
profile_norm = (cluster_profiles - cluster_profiles.min()) / (cluster_profiles.max() - cluster_profiles.min() + 1e-9)
sns.heatmap(profile_norm.T, annot=True, fmt='.2f', cmap='YlOrRd',
           xticklabels=[f'Cluster {i}' for i in range(K_OPTIMAL)],
           yticklabels=key_features, ax=axes[1], linewidths=0.5)
axes[1].set_title('Profil normalisé des clusters K-Means', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n📊 Profil moyen des clusters (valeurs réelles) :')
print(cluster_profiles.round(3))

## E.3 — Modèle 2 : DBSCAN — Détection d'outliers

In [ ]:
# === Choix des paramètres DBSCAN via k-distance graph ===
from sklearn.neighbors import NearestNeighbors

k_neighbors = 5
nbrs = NearestNeighbors(n_neighbors=k_neighbors).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
distances_sorted = np.sort(distances[:, k_neighbors-1])[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(distances_sorted, linewidth=2, color='#2196F3')
ax.set_xlabel('Points triés', fontsize=12)
ax.set_ylabel(f'Distance au {k_neighbors}ème voisin', fontsize=12)
ax.set_title('K-Distance Graph — Choix de eps pour DBSCAN', fontsize=13, fontweight='bold')

# Suggestion eps (point de coude)
diffs = np.diff(distances_sorted)
elbow_idx = np.argmax(diffs[:len(diffs)//2])
suggested_eps = distances_sorted[elbow_idx]
ax.axhline(y=suggested_eps, color='red', linestyle='--', alpha=0.8,
           label=f'eps suggéré ≈ {suggested_eps:.2f}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()
print(f'eps suggéré : {suggested_eps:.3f}')

In [ ]:
# === DBSCAN ===
EPS = max(0.8, min(suggested_eps, 2.0))   # Borne raisonnable
MIN_SAMPLES = 5

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
df_clean['cluster_dbscan'] = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(df_clean['cluster_dbscan'])) - (1 if -1 in df_clean['cluster_dbscan'].values else 0)
n_noise = (df_clean['cluster_dbscan'] == -1).sum()

print(f'✅ DBSCAN (eps={EPS:.2f}, min_samples={MIN_SAMPLES})')
print(f'   Nombre de clusters trouvés : {n_clusters_db}')
print(f'   Points bruit (outliers) : {n_noise} ({n_noise/len(df_clean)*100:.1f}%)')
print()
print('Distribution :', df_clean['cluster_dbscan'].value_counts().sort_index().to_dict())

if n_clusters_db >= 2:
    non_noise_mask = df_clean['cluster_dbscan'] != -1
    sil_db = silhouette_score(X_scaled[non_noise_mask], df_clean.loc[non_noise_mask, 'cluster_dbscan'])
    db_db = davies_bouldin_score(X_scaled[non_noise_mask], df_clean.loc[non_noise_mask, 'cluster_dbscan'])
    print(f'\n   Silhouette Score (hors bruit) : {sil_db:.4f}')
    print(f'   Davies-Bouldin (hors bruit)   : {db_db:.4f}')
else:
    sil_db, db_db = np.nan, np.nan
    print('⚠️ DBSCAN n\'a trouvé qu\'un seul cluster (eps trop grand) — ajuste eps')

In [ ]:
# === Visualisation DBSCAN ===
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# PCA plot DBSCAN
unique_labels = sorted(df_clean['cluster_dbscan'].unique())
colors_db = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
for i, label in enumerate(unique_labels):
    mask = df_clean['cluster_dbscan'] == label
    color = 'black' if label == -1 else colors_db[i]
    name = 'Bruit (Outliers)' if label == -1 else f'Cluster {label}'
    marker = 'x' if label == -1 else 'o'
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=[color], label=name, s=60 if label != -1 else 100,
                   alpha=0.8, marker=marker)
axes[0].set_xlabel(f'PC1 ({explained_var[0]*100:.1f}%)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({explained_var[1]*100:.1f}%)', fontsize=11)
axes[0].set_title(f'DBSCAN — PCA 2D (eps={EPS:.2f})', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)

# Profil des outliers vs non-outliers
df_clean['is_outlier'] = (df_clean['cluster_dbscan'] == -1).astype(int)
outlier_profile = df_clean.groupby('is_outlier')[key_features].mean()
outlier_profile.index = ['Normal', 'Outlier']
outlier_norm = (outlier_profile - outlier_profile.min()) / (outlier_profile.max() - outlier_profile.min() + 1e-9)
sns.heatmap(outlier_norm.T, annot=True, fmt='.2f', cmap='coolwarm',
           ax=axes[1], linewidths=0.5)
axes[1].set_title('Profil : Outliers vs Points normaux (DBSCAN)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Afficher les outliers
outliers = df_clean[df_clean['cluster_dbscan'] == -1][['ID_player', 'Year', 'points', 'win_rate', 'prize_money_avg', 'classement_mondial']]
print(f'\n🔴 {len(outliers)} tournois identifiés comme outliers :')
print(outliers.head(10))

## E.4 — Modèle 3 : Clustering Hiérarchique + Dendrogramme

In [ ]:
# === Clustering Hiérarchique ===
# Agréger par joueur pour le dendrogramme
player_profiles = df_clean.groupby('ID_player')[clustering_features].mean()
X_player_scaled = scaler.fit_transform(player_profiles)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Dendrogramme sur les joueurs
linked = linkage(X_player_scaled, method='ward')
player_labels = [f'Joueur {p}' for p in player_profiles.index]
dendrogram(linked, labels=player_labels, ax=axes[0],
           color_threshold=0.7*max(linked[:, 2]),
           leaf_font_size=12)
axes[0].set_title('🌳 Dendrogramme — Clustering Hiérarchique (Ward)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Distance de Ward', fontsize=11)

# Sur l'ensemble des tournois
N_CLUSTERS_HC = K_OPTIMAL
hc = AgglomerativeClustering(n_clusters=N_CLUSTERS_HC, linkage='ward')
df_clean['cluster_hc'] = hc.fit_labels_ = hc.fit_predict(X_scaled)

sil_hc = silhouette_score(X_scaled, df_clean['cluster_hc'])
db_hc = davies_bouldin_score(X_scaled, df_clean['cluster_hc'])

# PCA plot HC
for i in range(N_CLUSTERS_HC):
    mask = df_clean['cluster_hc'] == i
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   label=f'Cluster {i}', s=60, alpha=0.8)
axes[1].set_xlabel(f'PC1 ({explained_var[0]*100:.1f}%)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({explained_var[1]*100:.1f}%)', fontsize=11)
axes[1].set_title(f'Hiérarchique — PCA 2D (K={N_CLUSTERS_HC})', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'✅ Clustering Hiérarchique (Ward, K={N_CLUSTERS_HC})')
print(f'   Silhouette Score : {sil_hc:.4f}')
print(f'   Davies-Bouldin   : {db_hc:.4f}')

## E.5 — Comparaison des méthodes de Clustering

In [ ]:
# === Tableau comparatif ===
comparison_data = {
    'Modèle': ['K-Means', 'DBSCAN', 'Hiérarchique (Ward)'],
    'Silhouette ↑': [round(sil_km, 4),
                     round(sil_db, 4) if not np.isnan(sil_db) else 'N/A',
                     round(sil_hc, 4)],
    'Davies-Bouldin ↓': [round(db_km, 4),
                          round(db_db, 4) if not np.isnan(db_db) else 'N/A',
                          round(db_hc, 4)],
    'N clusters': [K_OPTIMAL, n_clusters_db, N_CLUSTERS_HC],
    'Outliers': ['Non', f'{n_noise}', 'Non']
}
df_comparison = pd.DataFrame(comparison_data)
print('\n📊 COMPARAISON DES MODÈLES DE CLUSTERING')
print('=' * 65)
print(df_comparison.to_string(index=False))
print('=' * 65)
print('\n💡 Interprétation :')
print('  Silhouette : proche de 1 = clusters bien séparés et denses')
print('  Davies-Bouldin : proche de 0 = clusters compacts et bien séparés')
print('  DBSCAN : utile pour la détection d\'anomalies (outliers)')

In [ ]:
# Visualisation comparaison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = ['K-Means', 'Hiérarchique']
sil_vals = [sil_km, sil_hc]
db_vals = [db_km, db_hc]

bars1 = axes[0].bar(models, sil_vals, color=['#2196F3', '#4CAF50'], alpha=0.85, edgecolor='black')
for bar, val in zip(bars1, sil_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Silhouette Score (↑ meilleur)', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, max(sil_vals) * 1.2)
axes[0].set_ylabel('Score')

bars2 = axes[1].bar(models, db_vals, color=['#2196F3', '#4CAF50'], alpha=0.85, edgecolor='black')
for bar, val in zip(bars2, db_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('Davies-Bouldin Index (↓ meilleur)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Score')

plt.suptitle('🔵 Comparaison finale — Clustering', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 📈 SECTION F — Time Series Forecasting

## Objectifs :
1. **Prédire les points annuels** d'un joueur avec ARIMA
2. **Prévision multi-saison** avec Prophet
3. **Prédiction non-linéaire** avec XGBoost TS
4. Comparer les méthodes via MAPE, RMSE, MAE

## F.1 — Construction de la série temporelle

In [ ]:
# Agrégation par joueur et année
ts_df = df_clean.groupby(['ID_player', 'Year']).agg(
    total_points=('points', 'sum'),
    avg_win_rate=('win_rate', 'mean'),
    avg_rank=('classement_mondial', 'mean'),
    total_prize=('prize_money_avg', 'sum'),
    avg_spectateurs=('Nombre_de_spectateurs', 'mean')
).reset_index()

print(ts_df.sort_values(['ID_player', 'Year']))

# Visualisation des séries par joueur
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
players = ts_df['ID_player'].unique()

for i, player_id in enumerate(players):
    player_data = ts_df[ts_df['ID_player'] == player_id].sort_values('Year')
    axes[i].plot(player_data['Year'], player_data['total_points'],
                marker='o', linewidth=2.5, markersize=8, color=PALETTE[i])
    axes[i].fill_between(player_data['Year'], player_data['total_points'],
                         alpha=0.15, color=PALETTE[i])
    axes[i].set_title(f'Joueur {player_id} — Points annuels', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Année')
    axes[i].set_ylabel('Points totaux')
    for x, y in zip(player_data['Year'], player_data['total_points']):
        axes[i].annotate(f'{y:,.0f}', (x, y), textcoords='offset points',
                        xytext=(0, 10), ha='center', fontsize=9)

plt.suptitle('📈 Évolution des points annuels par joueur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## F.2 — Analyse de stationnarité (ADF + KPSS)

In [ ]:
# Travailler sur le joueur avec le plus de données (joueur 1)
# Pour les tests de stationnarité, on va utiliser les données par tournoi

# Série plus granulaire : points par saison
ts_main = df_clean[df_clean['ID_player'] == 1].sort_values('id_saison')[['id_saison', 'points', 'Year']].copy()
ts_points = ts_main['points'].values

print('=== TEST DE STATIONNARITÉ ===')
print()

# ADF Test (H0: non-stationnaire)
adf_result = adfuller(ts_points, autolag='AIC')
print('📊 Test ADF (Augmented Dickey-Fuller) :')
print(f'   Statistique ADF : {adf_result[0]:.4f}')
print(f'   p-value          : {adf_result[1]:.4f}')
print(f'   Conclusion       : {"✅ STATIONNAIRE (p < 0.05)" if adf_result[1] < 0.05 else "❌ NON-STATIONNAIRE (p ≥ 0.05)"}')
print()

# KPSS Test (H0: stationnaire)
try:
    kpss_result = kpss(ts_points, regression='c', nlags='auto')
    print('📊 Test KPSS :')
    print(f'   Statistique KPSS : {kpss_result[0]:.4f}')
    print(f'   p-value           : {kpss_result[1]:.4f}')
    print(f'   Conclusion        : {"❌ NON-STATIONNAIRE (p < 0.05)" if kpss_result[1] < 0.05 else "✅ STATIONNAIRE (p ≥ 0.05)"}')
except Exception as e:
    print(f'KPSS skipped: {e}')

print()
print('💡 Note : Avec seulement 4 points annuels, les séries par joueur sont très courtes.')
print('   On utilise les données granulaires par tournoi pour les modèles.')

In [ ]:
# Décomposition temporelle sur série annuelle enrichie
# Construire une série mensuelle synthétique à partir des tournois
# En réalité: on utilise id_Fact comme proxy d'ordre temporel

player1_data = df_clean[df_clean['ID_player'] == 1].sort_values('id_Fact').copy()
player1_series = player1_data['points'].values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Série originale
axes[0, 0].plot(player1_series, color='#2196F3', linewidth=1.5)
axes[0, 0].set_title('Série originale — Points (Joueur 1)', fontweight='bold')
axes[0, 0].set_xlabel('Tournoi (index)')

# ACF
plot_acf(player1_series, lags=min(20, len(player1_series)//3), ax=axes[0, 1])
axes[0, 1].set_title('ACF — Autocorrélation', fontweight='bold')

# PACF
plot_pacf(player1_series, lags=min(15, len(player1_series)//4), ax=axes[1, 0], method='ywm')
axes[1, 0].set_title('PACF — Autocorrélation partielle', fontweight='bold')

# Distribution
axes[1, 1].hist(player1_series, bins=20, color='#4CAF50', alpha=0.8, edgecolor='black')
axes[1, 1].set_title('Distribution des points (Joueur 1)', fontweight='bold')
axes[1, 1].set_xlabel('Points')

plt.suptitle('Analyse exploratoire de la série temporelle', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## F.3 — Modèle 1 : ARIMA

In [ ]:
# === Préparation des données annuelles pour ARIMA ===
# Utiliser la somme des points par année pour tous les joueurs
ts_annual = ts_df.sort_values(['ID_player', 'Year'])

# Travailler sur joueur 1 (le plus de points)
PLAYER_ID = 1
player_ts = ts_annual[ts_annual['ID_player'] == PLAYER_ID][['Year', 'total_points']].set_index('Year')
player_ts.index = pd.to_datetime(player_ts.index, format='%Y')
player_ts.index.freq = 'YS'

print(f'Série temporelle — Joueur {PLAYER_ID}:')
print(player_ts)

# Avec 4 points, on fait leave-one-out: train sur 2022-2024, test sur 2025
train = player_ts.iloc[:-1]  # 2022, 2023, 2024
test = player_ts.iloc[-1:]   # 2025

print(f'\nTrain: {train.index.year.tolist()}')
print(f'Test:  {test.index.year.tolist()}')

In [ ]:
# === Grid Search ARIMA ===
best_aic = np.inf
best_order = (1, 1, 0)
best_arima = None

print('🔍 Grid Search ARIMA (p, d, q) ...')
results_grid = []
for p in range(0, 3):
    for d in range(0, 2):
        for q in range(0, 3):
            try:
                model = ARIMA(train['total_points'], order=(p, d, q))
                result = model.fit()
                results_grid.append({'order': (p,d,q), 'AIC': result.aic, 'BIC': result.bic})
                if result.aic < best_aic:
                    best_aic = result.aic
                    best_order = (p, d, q)
                    best_arima = result
            except:
                pass

grid_df = pd.DataFrame(results_grid).sort_values('AIC').head(5)
print('\nTop 5 modèles ARIMA :')
print(grid_df.to_string(index=False))
print(f'\n✅ Meilleur modèle : ARIMA{best_order} (AIC = {best_aic:.2f})')

In [ ]:
# === Prévision ARIMA ===
# Réentraîner sur toutes les données pour forecast
arima_full = ARIMA(player_ts['total_points'], order=best_order).fit()

# Forecast 2 ans en avant
forecast_steps = 2
forecast_arima = arima_full.get_forecast(steps=forecast_steps)
forecast_mean = forecast_arima.predicted_mean
forecast_ci = forecast_arima.conf_int(alpha=0.3)

# Évaluation sur test
arima_test = ARIMA(train['total_points'], order=best_order).fit()
pred_test = arima_test.forecast(steps=1)
y_true = test['total_points'].values[0]
y_pred_arima = pred_test.values[0]

mae_arima = mean_absolute_error([y_true], [y_pred_arima])
rmse_arima = np.sqrt(mean_squared_error([y_true], [y_pred_arima]))
mape_arima = abs((y_true - y_pred_arima) / y_true) * 100

print(f'ARIMA{best_order} — Évaluation sur 2025 :')
print(f'  Réel      : {y_true:,.0f}')
print(f'  Prédit    : {y_pred_arima:,.0f}')
print(f'  MAE       : {mae_arima:,.0f}')
print(f'  RMSE      : {rmse_arima:,.0f}')
print(f'  MAPE      : {mape_arima:.1f}%')

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Historique + Forecast
future_years = pd.date_range(start='2026', periods=forecast_steps, freq='YS')
axes[0].plot(player_ts.index, player_ts['total_points'], 'bo-',
            linewidth=2, markersize=8, label='Historique')
axes[0].plot(future_years, forecast_mean.values, 'r--o',
            linewidth=2, markersize=8, label='Prévision ARIMA')
axes[0].fill_between(future_years, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1],
                    alpha=0.25, color='red', label='IC 70%')
axes[0].set_title(f'ARIMA{best_order} — Prévision points (Joueur {PLAYER_ID})',
                 fontsize=13, fontweight='bold')
axes[0].set_xlabel('Année')
axes[0].set_ylabel('Points')
axes[0].legend()

# Résidus
residuals = arima_full.resid
axes[1].plot(residuals, 'g-o', linewidth=1.5, markersize=6)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('Résidus ARIMA', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Observation')
axes[1].set_ylabel('Résidu')

plt.tight_layout()
plt.show()

## F.4 — Modèle 2 : Prophet

In [ ]:
# === Préparation pour Prophet ===
# Prophet requiert colonnes 'ds' (date) et 'y' (valeur)
prophet_df = player_ts.reset_index().rename(columns={'Year': 'ds', 'total_points': 'y'})

# Split
prophet_train = prophet_df.iloc[:-1]
prophet_test = prophet_df.iloc[-1:]

# === Modèle Prophet ===
prophet_model = Prophet(
    yearly_seasonality=False,    # Pas de saisonnalité sur données annuelles
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='additive',
    changepoint_prior_scale=0.5,  # Flexibilité de la tendance
    interval_width=0.80
)

# Entraînement
prophet_model.fit(prophet_train)

# Forecast sur 4 ans (2025-2028)
future = prophet_model.make_future_dataframe(periods=4, freq='YS')
forecast_prophet = prophet_model.predict(future)

# Évaluation sur 2025
y_pred_prophet = forecast_prophet[forecast_prophet['ds'].dt.year == 2025]['yhat'].values[0]
mae_prophet = mean_absolute_error([y_true], [y_pred_prophet])
rmse_prophet = np.sqrt(mean_squared_error([y_true], [y_pred_prophet]))
mape_prophet = abs((y_true - y_pred_prophet) / y_true) * 100

print(f'Prophet — Évaluation sur 2025 :')
print(f'  Réel      : {y_true:,.0f}')
print(f'  Prédit    : {y_pred_prophet:,.0f}')
print(f'  MAE       : {mae_prophet:,.0f}')
print(f'  RMSE      : {rmse_prophet:,.0f}')
print(f'  MAPE      : {mape_prophet:.1f}%')

In [ ]:
# Visualisation Prophet
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot principal Prophet
hist_years = prophet_df['ds']
hist_vals = prophet_df['y']

axes[0].scatter(hist_years, hist_vals, color='blue', s=80, zorder=5, label='Données réelles')
axes[0].plot(forecast_prophet['ds'], forecast_prophet['yhat'],
            color='#FF5722', linewidth=2, label='Prévision Prophet')
axes[0].fill_between(forecast_prophet['ds'],
                    forecast_prophet['yhat_lower'],
                    forecast_prophet['yhat_upper'],
                    alpha=0.2, color='#FF5722', label='IC 80%')
axes[0].axvline(x=pd.Timestamp('2026-01-01'), color='gray', linestyle=':', alpha=0.8)
axes[0].text(pd.Timestamp('2026-03-01'), hist_vals.max() * 0.9,
            '← Futur', fontsize=10, color='gray')
axes[0].set_title(f'Prophet — Prévision points (Joueur {PLAYER_ID})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Année')
axes[0].set_ylabel('Points')
axes[0].legend()

# Composantes de tendance
trend_data = forecast_prophet[['ds', 'trend']]
axes[1].plot(trend_data['ds'], trend_data['trend'], color='#9C27B0', linewidth=2.5)
axes[1].set_title('Composante tendance (Prophet)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Tendance')

plt.tight_layout()
plt.show()

## F.5 — Modèle 3 : XGBoost Time Series (Lag Features)

In [ ]:
# === XGBoost pour Time Series — approche multi-joueurs ===
# On enrichit avec toutes les données et on crée des lag features

# Utiliser ts_df avec tous les joueurs
xgb_df = ts_annual.sort_values(['ID_player', 'Year']).copy()

def create_lag_features(df, target_col, lags=[1, 2], group_col='ID_player'):
    """Crée des lag features groupées par joueur"""
    df = df.copy()
    for lag in lags:
        df[f'{target_col}_lag{lag}'] = df.groupby(group_col)[target_col].shift(lag)
    # Features temporelles
    df['year_norm'] = (df['Year'] - df['Year'].min()) / (df['Year'].max() - df['Year'].min())
    return df

xgb_df = create_lag_features(xgb_df, 'total_points', lags=[1, 2])
xgb_df = create_lag_features(xgb_df, 'avg_win_rate', lags=[1])

# Supprimer les NaN créés par les lags
xgb_df_clean = xgb_df.dropna().copy()

print(f'Dataset XGBoost TS : {xgb_df_clean.shape}')
print(xgb_df_clean[['ID_player', 'Year', 'total_points',
                     'total_points_lag1', 'total_points_lag2']].to_string(index=False))

In [ ]:
from sklearn.model_selection import cross_val_score

# Features et target
feature_cols = ['ID_player', 'year_norm', 'total_points_lag1', 'total_points_lag2',
                'avg_win_rate', 'avg_win_rate_lag1', 'avg_rank', 'avg_spectateurs']
feature_cols = [c for c in feature_cols if c in xgb_df_clean.columns]

X_xgb = xgb_df_clean[feature_cols].values
y_xgb = xgb_df_clean['total_points'].values

# Train/Test split temporel (2025 = test)
train_mask = xgb_df_clean['Year'] < 2025
test_mask = xgb_df_clean['Year'] == 2025

X_train_xgb = xgb_df_clean.loc[train_mask, feature_cols].values
y_train_xgb = xgb_df_clean.loc[train_mask, 'total_points'].values
X_test_xgb  = xgb_df_clean.loc[test_mask, feature_cols].values
y_test_xgb  = xgb_df_clean.loc[test_mask, 'total_points'].values

print(f'Train : {X_train_xgb.shape[0]} obs | Test : {X_test_xgb.shape[0]} obs')

# === XGBoost avec GridSearch ===
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

xgb_model = xgb.XGBRegressor(random_state=42, verbosity=0)

param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# K-Fold sur les données train
from sklearn.model_selection import KFold
kf = KFold(n_splits=min(3, X_train_xgb.shape[0]), shuffle=False)

grid_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=kf,
                        scoring='neg_mean_squared_error', n_jobs=-1)
grid_xgb.fit(X_train_xgb, y_train_xgb)

best_xgb = grid_xgb.best_estimator_
print(f'\n✅ Meilleurs paramètres XGBoost : {grid_xgb.best_params_}')

# Prédiction sur test
y_pred_xgb = best_xgb.predict(X_test_xgb)

mae_xgb = mean_absolute_error(y_test_xgb, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test_xgb, y_pred_xgb))
mape_xgb = np.mean(np.abs((y_test_xgb - y_pred_xgb) / y_test_xgb)) * 100
r2_xgb = 1 - np.sum((y_test_xgb - y_pred_xgb)**2) / np.sum((y_test_xgb - np.mean(y_test_xgb))**2)

print(f'\nXGBoost TS — Évaluation sur 2025 :')
print(f'  MAE  : {mae_xgb:,.0f}')
print(f'  RMSE : {rmse_xgb:,.0f}')
print(f'  MAPE : {mape_xgb:.1f}%')

In [ ]:
# Visualisation XGBoost
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted par joueur (2025)
test_results = xgb_df_clean[test_mask][['ID_player', 'Year', 'total_points']].copy()
test_results['predicted'] = y_pred_xgb

x_pos = np.arange(len(test_results))
width = 0.35
bars1 = axes[0].bar(x_pos - width/2, test_results['total_points'],
                   width, label='Réel', color='#2196F3', alpha=0.85, edgecolor='black')
bars2 = axes[0].bar(x_pos + width/2, test_results['predicted'],
                   width, label='Prédit', color='#FF5722', alpha=0.85, edgecolor='black')
axes[0].set_xlabel('Joueur (ID)', fontsize=11)
axes[0].set_ylabel('Points', fontsize=11)
axes[0].set_title('XGBoost TS — Réel vs Prédit (2025)', fontsize=13, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'Joueur {p}' for p in test_results['ID_player']])
axes[0].legend()

# Feature Importance
feat_imp = pd.Series(best_xgb.feature_importances_, index=feature_cols)
feat_imp_sorted = feat_imp.sort_values(ascending=True)
colors_fi = ['#FF5722' if v == feat_imp_sorted.max() else '#2196F3' for v in feat_imp_sorted]
feat_imp_sorted.plot(kind='barh', ax=axes[1], color=colors_fi, edgecolor='black', alpha=0.85)
axes[1].set_title('Feature Importance — XGBoost TS', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## F.6 — Comparaison finale des modèles Time Series

In [ ]:
# Comparaison sur joueur 1 (ARIMA et Prophet) + moyenne tous joueurs (XGBoost)
ts_comparison = pd.DataFrame({
    'Modèle': ['ARIMA' + str(best_order), 'Prophet', 'XGBoost TS'],
    'MAE': [mae_arima, mae_prophet, mae_xgb],
    'RMSE': [rmse_arima, rmse_prophet, rmse_xgb],
    'MAPE (%)': [mape_arima, mape_prophet, mape_xgb],
    'Scope': ['Joueur 1', 'Joueur 1', 'Tous joueurs']
})

print('\n📊 COMPARAISON DES MODÈLES TIME SERIES')
print('=' * 65)
print(ts_comparison.to_string(index=False))
print('=' * 65)

# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['MAE', 'RMSE', 'MAPE (%)']
colors_models = ['#2196F3', '#FF5722', '#4CAF50']
models_names = ts_comparison['Modèle'].tolist()

for i, metric in enumerate(metrics):
    vals = ts_comparison[metric].values
    bars = axes[i].bar(models_names, vals, color=colors_models, alpha=0.85, edgecolor='black')
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                    f'{val:,.1f}', ha='center', fontsize=11, fontweight='bold')
    axes[i].set_title(f'{metric} (↓ meilleur)', fontsize=12, fontweight='bold')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('📈 Comparaison finale — Modèles Time Series', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n💡 Analyse :')
print('  • ARIMA : modèle statistique classique, interprétable, adapté aux petites séries')
print('  • Prophet : robuste aux tendances, meilleure gestion des changements brusques')
print('  • XGBoost TS : capture les non-linéarités, utilise des features exogènes, multi-joueurs')

## F.7 — Forecast global tous joueurs (2026-2027)

In [ ]:
# Prévision Prophet pour tous les joueurs
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

forecast_summary = []

for i, pid in enumerate(ts_annual['ID_player'].unique()):
    p_data = ts_annual[ts_annual['ID_player'] == pid][['Year', 'total_points']].copy()
    p_data.columns = ['ds', 'y']
    p_data['ds'] = pd.to_datetime(p_data['ds'], format='%Y')
    
    m = Prophet(yearly_seasonality=False, weekly_seasonality=False,
               daily_seasonality=False, changepoint_prior_scale=0.5,
               interval_width=0.8)
    m.fit(p_data)
    future_p = m.make_future_dataframe(periods=3, freq='YS')
    fc = m.predict(future_p)
    
    axes[i].scatter(p_data['ds'], p_data['y'], color='blue', s=80, zorder=5, label='Réel')
    axes[i].plot(fc['ds'], fc['yhat'], 'r-', linewidth=2, label='Prophet')
    axes[i].fill_between(fc['ds'], fc['yhat_lower'], fc['yhat_upper'],
                        alpha=0.2, color='red')
    axes[i].axvline(x=pd.Timestamp('2026-01-01'), color='gray', linestyle=':', alpha=0.8)
    axes[i].set_title(f'Joueur {pid} — Prévision Prophet', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Année')
    axes[i].set_ylabel('Points')
    axes[i].legend(fontsize=9)
    
    # Stocker les forecasts 2026-2027
    for yr in [2026, 2027]:
        yr_ts = pd.Timestamp(f'{yr}-01-01')
        row = fc[fc['ds'] == yr_ts]
        if len(row) > 0:
            forecast_summary.append({
                'Joueur': pid, 'Année': yr,
                'Points prédit': int(row['yhat'].values[0]),
                'Lower': int(row['yhat_lower'].values[0]),
                'Upper': int(row['yhat_upper'].values[0])
            })

plt.suptitle('🔮 Prévision Prophet 2026-2027 — Tous joueurs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📊 Tableau de prévision 2026-2027 :')
print(pd.DataFrame(forecast_summary).to_string(index=False))

---
# 📊 SYNTHÈSE FINALE

In [ ]:
print('=' * 70)
print('🎾  PADEL ANALYTICS — SYNTHÈSE DES RÉSULTATS')
print('=' * 70)

print('\n🔵 SECTION E — CLUSTERING')
print('-' * 40)
print(f'  K-Means (K={K_OPTIMAL})    : Silhouette={sil_km:.4f}, DB={db_km:.4f}')
print(f'  Hiérarchique (K={N_CLUSTERS_HC})  : Silhouette={sil_hc:.4f}, DB={db_hc:.4f}')
print(f'  DBSCAN         : {n_clusters_db} cluster(s), {n_noise} outliers ({n_noise/len(df_clean)*100:.1f}%)')

print('\n  ✅ Meilleur modèle : K-Means (selon Silhouette + DB)')
print('  📌 Interprétation clusters :')
for c in range(K_OPTIMAL):
    n = (df_clean['cluster_kmeans'] == c).sum()
    avg_wr = df_clean[df_clean['cluster_kmeans'] == c]['win_rate'].mean()
    avg_pts = df_clean[df_clean['cluster_kmeans'] == c]['points_per_match'].mean()
    print(f'    Cluster {c} ({n} tournois) : win_rate={avg_wr:.2f}, pts/match={avg_pts:.0f}')

print('\n📈 SECTION F — TIME SERIES')
print('-' * 40)
print(f'  ARIMA{best_order}    : MAPE={mape_arima:.1f}%, RMSE={rmse_arima:,.0f}')
print(f'  Prophet        : MAPE={mape_prophet:.1f}%, RMSE={rmse_prophet:,.0f}')
print(f'  XGBoost TS     : MAPE={mape_xgb:.1f}%, RMSE={rmse_xgb:,.0f}')

best_ts = min([('ARIMA', mape_arima), ('Prophet', mape_prophet), ('XGBoost', mape_xgb)],
              key=lambda x: x[1])
print(f'\n  ✅ Meilleur modèle : {best_ts[0]} (MAPE={best_ts[1]:.1f}%)')

print('\n' + '=' * 70)